# Hierarchical Memory Layers

> **Organize agent memory into speed tiers (L1/L2/L3) with automatic promotion and demotion, like a CPU cache hierarchy.**

Think of your kitchen. Salt and pepper sit right next to the stove. You grab them in under a second. Less common spices go in a cabinet above the counter. They take a few seconds to reach, but the cabinet holds dozens of jars. Bulk ingredients live in the pantry down the hall. It takes the longest to walk there, but it can store everything. You'd never keep 50 jars on the counter. You'd also never walk to the pantry for salt every time you cook.

This arrangement works because items move based on usage. You pull the cumin to the counter during taco week. You push the holiday baking spices back to the pantry in January. The system adapts.

LLM agents face the same problem. The context window is fast but small. A vector database offers large capacity with millisecond retrieval. Archival storage provides near-unlimited space at higher latency (slower response time). Without tiering, agents either stuff everything into context (wasting tokens) or query a single store for every fact (adding latency even for things the agent needs on every turn).

**Hierarchical Memory Layers** split storage into three tiers:

| Tier | Analogy | Speed | Capacity |
|------|---------|-------|----------|
| **L1** (Hot) | Counter spices | Instant (in context) | Small (token-limited) |
| **L2** (Warm) | Cabinet spices | Fast (vector search) | Medium (100k+ items) |
| **L3** (Cold) | Pantry bulk | Slower (archive query) | Unlimited |

Promotion and demotion policies move memories between tiers based on access patterns. A fact the agent uses every turn (like the user's name) lives in L1. A relevant but less frequent fact (past project details) stays in L2. A rarely needed record (a conversation from months ago) rests in L3.

**What you'll build in this notebook:**
1. A `TieredMemory` data model with access tracking metadata.
2. A `HierarchicalMemoryManager` with three storage tiers.
3. Cascading retrieval (L1 first, then L2, then L3).
4. Automatic promotion and demotion based on access frequency.
5. A conversational agent that uses all three tiers.

**By the end you'll understand:**
- How to split memory into tiers with different speed and cost profiles.
- How promotion and demotion policies keep the right data in the right tier.
- When this approach outperforms flat memory, and when it adds unnecessary complexity.

## Key Concepts

- **L1 / Context tier**: The agent's context window. Fastest access (zero retrieval latency), smallest capacity. Holds memories the agent needs on every turn.
- **L2 / Warm tier**: A vector database (a database that finds items by meaning, not exact keywords). Offers sub-second semantic retrieval over a large corpus.
- **L3 / Cold tier**: Archival storage (a SQL database, document store, or file archive). Largest capacity, highest retrieval cost. Holds historical records and rarely accessed data.
- **Promotion**: Moving a memory from a slower tier to a faster one. Triggered when access frequency exceeds a threshold.
- **Demotion (eviction)**: Moving a memory from a faster tier to a slower one. Triggered by staleness (time since last access) or capacity pressure.
- **Access frequency**: How often a memory is retrieved within a recent window. This counter drives promotion and demotion decisions.
- **Cascading retrieval**: Searching L1 first, falling through to L2 on a miss, then L3. This mirrors how CPU cache misses cascade through the hierarchy.
- **Embedding**: A list of numbers that captures the meaning of a text. Two texts with similar meanings produce embeddings that are close together in vector space.
- **Cosine similarity**: A measure of how similar two embeddings are. Values range from -1 (opposite) to 1 (identical meaning).

## Architecture

<p align="center">
 <img src="../../images/diagrams/13_hierarchical_memory_layers.svg" alt="Hierarchical Memory Layers Architecture" width="720"/>
</p>

<details><summary>Mermaid source</summary>

```mermaid
flowchart TB
 Q["Query"] --> L1

 subgraph L1["L1: Context Window"]
 direction LR
 L1D["Fastest | Smallest\n~4k-16k tokens"]
 end

 L1 -->|"Cache Miss"| L2

 subgraph L2["L2: Vector DB"]
 direction LR
 L2D["Fast | Medium\n~100k+ memories"]
 end

 L2 -->|"Cache Miss"| L3

 subgraph L3["L3: Archive DB"]
 direction LR
 L3D["Slow | Largest\nUnlimited capacity"]
 end

 L2 -->|"Promote\n(high access freq)"| L1
 L1 -->|"Demote\n(stale / low freq)"| L2
 L3 -->|"Promote\n(re-accessed)"| L2
 L2 -->|"Demote\n(cold / aged)"| L3

 AC["Access Counter"] -.->|"Feeds"| L1
 AC -.->|"Feeds"| L2
 AC -.->|"Feeds"| L3

 style L1 fill:#2d5a2d,stroke:#4a9,color:#fff
 style L2 fill:#5a5a2d,stroke:#aa4,color:#fff
 style L3 fill:#5a2d2d,stroke:#a44,color:#fff
```

</details>

**Data flow**: A query first hits **L1** (the context window). On a miss, it cascades to **L2** (vector DB) and then **L3** (archival storage).

The **Access Counter** tracks hit frequency across all tiers. When a memory in L2 gets accessed frequently (above a promotion threshold), it moves up to L1. When an L1 memory goes stale or L1 reaches capacity, it moves down to L2. The same logic applies between L2 and L3.

This creates a self-organizing hierarchy that adapts to the agent's evolving information needs.

## Setup

Install dependencies and configure API access.

You'll need:
- `OPENAI_API_KEY` environment variable set (for embeddings and chat completions).
- A `.env` file in your working directory, or the variable exported in your shell.

In [ ]:
%pip install -q openai python-dotenv numpy

Import libraries and initialize the OpenAI client.

In [ ]:
import os
import time
import uuid
import json
from dataclasses import dataclass, field, asdict
from typing import Optional

import numpy as np
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI() # reads OPENAI_API_KEY from environment
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in your .env file"

EMBED_MODEL = "text-embedding-3-small"
CHAT_MODEL = "gpt-4o-mini"

print("Setup complete.")

## Implementation

We'll build the hierarchy in four steps:

1. **Embedding helper** for converting text to vectors.
2. **TieredMemory data model** with access tracking.
3. **HierarchicalMemoryManager** with three tiers and cascading retrieval.
4. **Promotion and demotion policies** that run between turns.

### Step 1: Embedding Helper

We need a function that converts text into an embedding (a list of numbers capturing meaning). Two memories about similar topics will produce embeddings that are close together. We'll use this for semantic search in L2.

In [ ]:
def get_embedding(text: str) -> list[float]:
 """Convert text to a vector embedding using OpenAI's API."""
 response = client.embeddings.create(
 model=EMBED_MODEL,
 input=text,
 )
 return response.data[0].embedding


def cosine_similarity(a: list[float], b: list[float]) -> float:
 """Compute cosine similarity between two vectors."""
 a_arr = np.array(a)
 b_arr = np.array(b)
 return float(np.dot(a_arr, b_arr) / (np.linalg.norm(a_arr) * np.linalg.norm(b_arr)))

### Step 2: TieredMemory Data Model

Each memory stores its content, embedding, current tier, and access metadata. The access count and timestamp drive promotion and demotion decisions.

In [ ]:
@dataclass
class TieredMemory:
 """A single memory item with tier placement and access tracking."""
 content: str
 memory_id: str = field(default_factory=lambda: str(uuid.uuid4()))
 embedding: list[float] = field(default_factory=list)
 tier: str = "L2" # L1, L2, or L3
 access_count: int = 0 # total times retrieved
 last_accessed: float = 0.0 # timestamp of last retrieval
 created_at: float = field(default_factory=time.time)

 def record_access(self) -> None:
 """Bump access counter and update timestamp."""
 self.access_count += 1
 self.last_accessed = time.time()

 def staleness(self) -> float:
 """Seconds since last access. Higher means more stale."""
 if self.last_accessed == 0.0:
 return time.time() - self.created_at
 return time.time() - self.last_accessed

 def summary(self) -> str:
 """Short display string."""
 preview = self.content[:60] + ("..." if len(self.content) > 60 else "")
 return f"[{self.tier}] (hits={self.access_count}) {preview}"

### Step 3: HierarchicalMemoryManager

This is the core class. It manages three tiers:

- **L1**: A Python dict (in-memory). Small, fast. These memories get injected into every LLM prompt.
- **L2**: A Python dict acting as our vector store. In production you'd swap this for Chroma, Pinecone, or Qdrant.
- **L3**: A Python dict acting as our archive. In production this would be a SQL database or blob store.

We use dicts here to keep the notebook self-contained. The retrieval and tier-movement logic stays the same regardless of backend.

In [ ]:
class HierarchicalMemoryManager:
 """Three-tier memory system with promotion and demotion."""

 def __init__(
 self,
 l1_capacity: int = 5,
 promote_threshold: int = 3,
 demote_staleness_seconds: float = 300.0,
 similarity_threshold: float = 0.3,
 ):
 # Tier storage: memory_id -> TieredMemory
 self.l1: dict[str, TieredMemory] = {}
 self.l2: dict[str, TieredMemory] = {}
 self.l3: dict[str, TieredMemory] = {}

 # Policy parameters
 self.l1_capacity = l1_capacity
 self.promote_threshold = promote_threshold # accesses needed for L2 -> L1
 self.demote_staleness = demote_staleness_seconds # seconds idle before L1 -> L2
 self.similarity_threshold = similarity_threshold # min cosine similarity for retrieval

 # Event log for visibility into tier movements
 self.event_log: list[str] = []

 # ── Store ─────────────────────────────────────────────────────
 def store(self, content: str, tier: str = "L2") -> TieredMemory:
 """Add a new memory. Default tier is L2 (warm)."""
 embedding = get_embedding(content)
 memory = TieredMemory(
 content=content,
 embedding=embedding,
 tier=tier,
 )
 self._place(memory)
 self._log(f"STORE -> {memory.tier}: {content[:50]}")
 return memory

 def _place(self, memory: TieredMemory) -> None:
 """Put a memory into its designated tier's storage."""
 tier_store = {"L1": self.l1, "L2": self.l2, "L3": self.l3}[memory.tier]
 tier_store[memory.memory_id] = memory

The `query` method implements cascading retrieval. It searches L1 first, then L2 if more results are needed, then L3. Each hit bumps the memory's access counter, which feeds the promotion logic later.

In [ ]:
 # ── Retrieve ──────────────────────────────────────────────────
 def query(self, query_text: str, top_k: int = 3) -> list[TieredMemory]:
 """Cascading retrieval: search L1, then L2, then L3."""
 query_emb = get_embedding(query_text)
 results: list[TieredMemory] = []

 # Search L1 first (exact in-context memories)
 l1_hits = self._search_tier(self.l1, query_emb, top_k)
 results.extend(l1_hits)

 # Fall through to L2 if we need more
 if len(results) < top_k:
 remaining = top_k - len(results)
 l2_hits = self._search_tier(self.l2, query_emb, remaining)
 results.extend(l2_hits)

 # Fall through to L3 if still short
 if len(results) < top_k:
 remaining = top_k - len(results)
 l3_hits = self._search_tier(self.l3, query_emb, remaining)
 results.extend(l3_hits)

 # Record access for each hit
 for mem in results:
 mem.record_access()

 return results

 def _search_tier(
 self,
 tier_store: dict[str, TieredMemory],
 query_emb: list[float],
 top_k: int,
 ) -> list[TieredMemory]:
 """Semantic search within a single tier."""
 if not tier_store:
 return []

 scored = []
 for mem in tier_store.values():
 sim = cosine_similarity(query_emb, mem.embedding)
 if sim >= self.similarity_threshold:
 scored.append((sim, mem))

 scored.sort(key=lambda x: x[0], reverse=True)
 return [mem for _, mem in scored[:top_k]]

The `_move` helper transfers a memory between tiers. The `run_maintenance` method is the promotion/demotion sweep. It checks four rules: promote hot L2 items to L1, demote stale L1 items to L2, demote cold L2 items to L3, and promote re-accessed L3 items back to L2. Call this between agent turns to keep tiers balanced.

In [ ]:
 # ── Tier movement ─────────────────────────────────────────────
 def _move(self, memory: TieredMemory, target_tier: str) -> None:
 """Move a memory from its current tier to target_tier."""
 source_store = {"L1": self.l1, "L2": self.l2, "L3": self.l3}[memory.tier]
 target_store = {"L1": self.l1, "L2": self.l2, "L3": self.l3}[target_tier]

 # Remove from source
 source_store.pop(memory.memory_id, None)

 # Update tier label and place
 old_tier = memory.tier
 memory.tier = target_tier
 target_store[memory.memory_id] = memory

 self._log(f"MOVE {old_tier} -> {target_tier}: {memory.content[:50]}")

 # ── Maintenance (promotion / demotion sweep) ──────────────────
 def run_maintenance(self) -> list[str]:
 """Run one promotion/demotion cycle. Call between agent turns."""
 actions: list[str] = []

 # 1. Promote hot L2 memories to L1
 for mem in list(self.l2.values()):
 if mem.access_count >= self.promote_threshold:
 if len(self.l1) < self.l1_capacity:
 self._move(mem, "L1")
 actions.append(f"Promoted to L1: {mem.content[:40]}")
 else:
 # L1 is full: evict the stalest L1 memory first
 stalest = max(self.l1.values(), key=lambda m: m.staleness())
 self._move(stalest, "L2")
 stalest.access_count = 0 # reset counter on demotion
 actions.append(f"Demoted from L1: {stalest.content[:40]}")
 self._move(mem, "L1")
 actions.append(f"Promoted to L1: {mem.content[:40]}")

 # 2. Demote stale L1 memories to L2
 for mem in list(self.l1.values()):
 if mem.staleness() > self.demote_staleness:
 self._move(mem, "L2")
 actions.append(f"Demoted stale L1 to L2: {mem.content[:40]}")

 # 3. Demote cold L2 memories to L3
 for mem in list(self.l2.values()):
 if mem.staleness() > self.demote_staleness * 3: # 3x slower to archive
 self._move(mem, "L3")
 actions.append(f"Demoted cold L2 to L3: {mem.content[:40]}")

 # 4. Promote re-accessed L3 memories back to L2
 for mem in list(self.l3.values()):
 if mem.access_count >= 1 and mem.staleness() < self.demote_staleness:
 self._move(mem, "L2")
 actions.append(f"Promoted L3 to L2: {mem.content[:40]}")

 return actions

These utility methods round out the manager. `get_l1_context` formats L1 memories as text for the LLM prompt. `tier_counts` and `all_memories` help with debugging and inspecting the hierarchy.

In [ ]:
 # ── Context builder (for LLM prompts) ─────────────────────────
 def get_l1_context(self) -> str:
 """Format L1 memories as text to inject into the LLM prompt."""
 if not self.l1:
 return "No pinned memories."
 lines = []
 for mem in self.l1.values():
 lines.append(f"- {mem.content}")
 return "\n".join(lines)

 # ── Utilities ─────────────────────────────────────────────────
 def tier_counts(self) -> dict[str, int]:
 return {"L1": len(self.l1), "L2": len(self.l2), "L3": len(self.l3)}

 def all_memories(self) -> list[TieredMemory]:
 return list(self.l1.values()) + list(self.l2.values()) + list(self.l3.values())

 def _log(self, message: str) -> None:
 self.event_log.append(message)

### Step 4: Conversational Agent with Hierarchical Memory

Now we wire the memory manager into a conversational agent. The agent:
1. Extracts key facts from each conversation turn using the LLM.
2. Stores extracted facts in L2 by default.
3. Queries all tiers for relevant context before responding.
4. Runs maintenance (promotion/demotion) after each turn.
5. Injects L1 memories into every prompt as "pinned" context.

In [ ]:
class HierarchicalMemoryAgent:
 """Conversational agent backed by three-tier memory."""

 def __init__(
 self,
 model: str = CHAT_MODEL,
 l1_capacity: int = 5,
 promote_threshold: int = 3,
 demote_staleness_seconds: float = 300.0,
 ):
 self.model = model
 self.memory = HierarchicalMemoryManager(
 l1_capacity=l1_capacity,
 promote_threshold=promote_threshold,
 demote_staleness_seconds=demote_staleness_seconds,
 )
 self.conversation: list[dict] = [] # recent turns for LLM context
 self.max_conversation_turns = 10 # keep last N turns in conversation

 def chat(self, user_input: str) -> str:
 """Process one user turn: retrieve, respond, extract, maintain."""
 # 1. Retrieve relevant memories from all tiers
 retrieved = self.memory.query(user_input, top_k=3)
 retrieved_text = self._format_retrieved(retrieved)

 # 2. Build the prompt with L1 context and retrieved memories
 l1_context = self.memory.get_l1_context()

 system_prompt = (
 "You are a helpful assistant with hierarchical memory. "
 "Use the pinned memories and retrieved context to give informed answers.\n\n"
 f"## Pinned Memories (always available)\n{l1_context}\n\n"
 f"## Retrieved Context (from search)\n{retrieved_text}\n\n"
 "Keep replies concise (2-3 sentences). "
 "If you recall something from memory, mention it."
 )

 self.conversation.append({"role": "user", "content": user_input})

 response = client.chat.completions.create(
 model=self.model,
 messages=[
 {"role": "system", "content": system_prompt},
 *self.conversation[-self.max_conversation_turns:],
 ],
 max_tokens=256,
 )

 assistant_reply = response.choices[0].message.content
 self.conversation.append({"role": "assistant", "content": assistant_reply})

 # 3. Extract and store new facts from this turn
 self._extract_and_store(user_input)

 # 4. Run maintenance (promotion/demotion)
 actions = self.memory.run_maintenance()

 return assistant_reply

The `_extract_and_store` method uses a separate LLM call to pull key facts from each user message. Each fact goes into L2 by default. Over time, frequently accessed facts get promoted to L1 by the maintenance sweep.

In [ ]:
 def _extract_and_store(self, user_input: str) -> None:
 """Use the LLM to pull key facts from the user's message."""
 extraction_response = client.chat.completions.create(
 model=self.model,
 messages=[
 {
 "role": "system",
 "content": (
 "Extract key facts from the user message. "
 "Return each fact on its own line. "
 "If there are no facts worth storing, return NONE. "
 "Do not include opinions or small talk."
 ),
 },
 {"role": "user", "content": user_input},
 ],
 max_tokens=200,
 )

 facts_text = extraction_response.choices[0].message.content.strip()
 if facts_text.upper() == "NONE" or not facts_text:
 return

 for line in facts_text.split("\n"):
 fact = line.strip().lstrip("- ").strip()
 if fact and len(fact) > 5:
 self.memory.store(fact, tier="L2")

 def _format_retrieved(self, memories: list[TieredMemory]) -> str:
 if not memories:
 return "No relevant memories found."
 lines = []
 for mem in memories:
 lines.append(f"- [{mem.tier}] {mem.content}")
 return "\n".join(lines)

## Example Run

Let's see the hierarchy in action. We'll:
1. Have a multi-turn conversation that stores facts across tiers.
2. Watch promotion happen as the agent accesses certain facts repeatedly.
3. Inspect the tier distribution after each exchange.

### Part 1: Seeding memories across tiers

First, let's pre-load some memories at different tiers to see cascading retrieval in action. In a real system, these would accumulate over time.

In [ ]:
manager = HierarchicalMemoryManager(
 l1_capacity=3,
 promote_threshold=2, # promote after 2 accesses
 demote_staleness_seconds=60, # short window for demo
)

# Store memories at different tiers
manager.store("User's name is Alice.", tier="L1") # pinned: always in context
manager.store("Alice is a machine learning engineer.", tier="L1")

manager.store("Alice works at Acme Corp.", tier="L2") # warm: needs retrieval
manager.store("Alice is building a recommendation system.", tier="L2")
manager.store("The project uses PyTorch and FAISS.", tier="L2")

manager.store("Alice attended MIT for undergrad.", tier="L3") # cold: archived
manager.store("Alice's first job was at a startup called DataMinds.", tier="L3")

print("Tier distribution:", manager.tier_counts())
print("\nAll memories:")
for mem in manager.all_memories():
 print(f" {mem.summary()}")

### Part 2: Cascading retrieval

Watch how a query searches L1 first, then cascades to L2 and L3.

In [ ]:
print("=" * 60)
print("Query: 'What does Alice work on?'")
print("=" * 60)

results = manager.query("What does Alice work on?", top_k=3)
for r in results:
 print(f" Found in {r.tier}: {r.content}")

print(f"\nQuery: 'Where did Alice go to school?'")
results = manager.query("Where did Alice go to school?", top_k=2)
for r in results:
 print(f" Found in {r.tier}: {r.content}")

print("\nEvent log:")
for event in manager.event_log:
 print(f" {event}")

### Part 3: Promotion in action

Now let's access the same L2 memory multiple times and watch it get promoted to L1. Remember, our `promote_threshold` is 2.

In [ ]:
print("Before promotion:")
print(f" Tier counts: {manager.tier_counts()}")
print()

# Access 'Alice works at Acme Corp' enough times to trigger promotion
for i in range(2):
 results = manager.query("Where does Alice work?", top_k=1)
 if results:
 print(f" Access {i+1}: '{results[0].content}' (hits={results[0].access_count}, tier={results[0].tier})")

# Run maintenance to apply promotion
actions = manager.run_maintenance()
print("\nMaintenance actions:")
for action in actions:
 print(f" {action}")

print(f"\nAfter promotion:")
print(f" Tier counts: {manager.tier_counts()}")
print("\nL1 contents:")
for mem in manager.l1.values():
 print(f" {mem.summary()}")

### Part 4: Full agent conversation

Now let's use the full `HierarchicalMemoryAgent` for a multi-turn conversation. The agent extracts facts, stores them in L2, and promotes frequently accessed ones to L1.

In [ ]:
agent = HierarchicalMemoryAgent(
 l1_capacity=3,
 promote_threshold=2,
 demote_staleness_seconds=120,
)

conversation_turns = [
 "Hi! I'm Bob and I'm a software engineer at CloudScale.",
 "I'm working on a distributed caching system using Redis.",
 "What do you know about me so far?",
 "I also have a dog named Pixel. He's a corgi.",
 "Tell me again: where do I work and what am I building?",
 "My team has 8 people and we ship weekly.",
 "Remind me what my dog's name is.",
]

for turn in conversation_turns:
 print(f"\n{'='*60}")
 print(f"User: {turn}")
 reply = agent.chat(turn)
 print(f"Agent: {reply}")
 counts = agent.memory.tier_counts()
 print(f" [Tiers: L1={counts['L1']}, L2={counts['L2']}, L3={counts['L3']}]")

Let's inspect the final state of each tier.

In [ ]:
print("Final memory state across all tiers:")
print(f"\nTotal memories: {sum(agent.memory.tier_counts().values())}")
print(f"Tier counts: {agent.memory.tier_counts()}")

for tier_name, tier_store in [("L1 (Hot)", agent.memory.l1),
 ("L2 (Warm)", agent.memory.l2),
 ("L3 (Cold)", agent.memory.l3)]:
 print(f"\n--- {tier_name} ---")
 if not tier_store:
 print(" (empty)")
 for mem in tier_store.values():
 print(f" {mem.summary()}")

print("\n--- Event Log (last 10) ---")
for event in agent.memory.event_log[-10:]:
 print(f" {event}")

### Part 5: Persistence

A production hierarchy must survive restarts. The data model is straightforward to serialize (convert to a file format). Here's a minimal JSON round-trip.

In [ ]:
def save_hierarchy(manager: HierarchicalMemoryManager, path: str) -> None:
 """Save all tiers to a JSON file."""
 data = {
 "l1": {mid: asdict(m) for mid, m in manager.l1.items()},
 "l2": {mid: asdict(m) for mid, m in manager.l2.items()},
 "l3": {mid: asdict(m) for mid, m in manager.l3.items()},
 }
 with open(path, "w") as f:
 json.dump(data, f, indent=2)
 total = len(manager.l1) + len(manager.l2) + len(manager.l3)
 print(f"Saved {total} memories across 3 tiers to {path}")


def load_hierarchy(path: str, **kwargs) -> HierarchicalMemoryManager:
 """Restore a HierarchicalMemoryManager from a JSON file."""
 with open(path) as f:
 data = json.load(f)
 mgr = HierarchicalMemoryManager(**kwargs)
 for tier_key, tier_store in [("l1", mgr.l1), ("l2", mgr.l2), ("l3", mgr.l3)]:
 for mid, mdata in data[tier_key].items():
 mem = TieredMemory(**{k: v for k, v in mdata.items()})
 tier_store[mid] = mem
 total = len(mgr.l1) + len(mgr.l2) + len(mgr.l3)
 print(f"Loaded {total} memories from {path}")
 return mgr


# Round-trip test
save_hierarchy(agent.memory, "hierarchy_snapshot.json")
restored = load_hierarchy("hierarchy_snapshot.json", l1_capacity=3, promote_threshold=2)
print(f"Restored tier counts: {restored.tier_counts()}")

Clean up temporary files.

In [ ]:
for f in ["hierarchy_snapshot.json"]:
 if os.path.exists(f):
 os.remove(f)
 print(f"Removed {f}")

## Tradeoffs

### When Hierarchical Memory Layers Work Well

- **Long-lived agents** that accumulate thousands of memories over weeks or months. The hierarchy keeps active facts fast and archives the rest.
- **Cost-sensitive deployments** where stuffing everything into the context window wastes tokens. L1 holds only what matters most, reducing input token counts.
- **Mixed-latency requirements**: critical facts (user name, current project) are always in context, while historical data is still reachable through cascading search.

### When It Breaks Down

- **Short conversations** (under 20 turns) rarely generate enough data for tiering to help. A flat list or sliding window works fine.
- **Tuning complexity**: promotion thresholds, staleness windows, and L1 capacity all need calibration. Bad settings can thrash memories between tiers or leave stale data pinned in L1.
- **Cold start problem**: a new agent has no access history. Everything sits in L2 until enough queries accumulate.
- **Storage overhead**: every memory needs an embedding (for semantic search) and access metadata. This adds latency and cost to every store operation.

### Production Considerations

- Replace the in-memory dicts with real backends: Redis or SQLite for L1 metadata, Chroma or Pinecone for L2, PostgreSQL or S3 for L3.
- Run maintenance asynchronously (in the background) so it doesn't block the response path.
- Add TTL-based (time-to-live) expiration for L3 to avoid unbounded storage growth.
- Monitor tier movement rates. High churn between L1 and L2 signals a poorly tuned promote threshold.

## Further Reading

- Patterson & Hennessy, [*Computer Organization and Design*](https://www.elsevier.com/books/computer-organization-and-design/patterson/978-0-12-820109-1?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques), 2017. The classic text on cache hierarchies that inspires this approach.
- Packer et al., ["MemGPT: Towards LLMs as Operating Systems"](https://arxiv.org/abs/2310.08560), 2023. Introduces virtual context management with explicit paging between main context and external storage.
- Zhang et al., ["A Survey on the Memory Mechanism of Large Language Model Based Agents"](https://arxiv.org/abs/2404.13501), 2024. Covers memory architectures including hierarchical approaches.
- Nuxoll & Laird, ["Extending Cognitive Architecture with Episodic Memory"](https://aaai.org/papers/01560-extending-cognitive-architecture-with-episodic-memory/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques), AAAI 2007. Explores episodic memory tiers within the Soar cognitive architecture.

---

*Previous: [12 - Working Memory & Context Window](../12_working_memory_context_window/) | Next: [14 - Memory Consolidation](../14_memory_consolidation/)*

## 🧪 Try It Yourself

Three small challenges to deepen your understanding. Each should take 10-30 minutes.

### Challenge 1: Custom promotion thresholds
Change the access-count thresholds for L3-to-L2 and L2-to-L1 promotion in `HierarchicalMemoryManager`. Try aggressive promotion (low thresholds) and conservative promotion (high thresholds). Measure the L1 cache hit rate for each setting over 30 queries.

### Challenge 2: Tier distribution over time
Store 50 memories and run `run_maintenance()` after every 10th query. After each maintenance pass, record how many memories sit in L1, L2, and L3. Plot the tier distribution over time as a stacked area chart.

### Challenge 3: Persistent tier storage
Extend `save_hierarchy()` and `load_hierarchy()` to write each tier to a separate file (or SQLite table). Reload the hierarchy in a new Python session and verify that queries still return correct results. This connects directly to the persistence patterns in 21 Cross-Session Memory.


![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--13-hierarchical-memory-layers--hierarchical-memory-layers)